# DantinoX — Colab Quickstart

> *"E quindi uscimmo a riveder le stelle."*

This notebook trains a decoder-only Transformer with **DantinoX** on a HuggingFace dataset, generates text, benchmarks throughput, and optionally pushes the checkpoint to the Hub.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or A100 if available).

| Step | What happens |
|---|---|
| **1 — Install** | JAX CUDA + DantinoX from PyPI |
| **2 — Check GPU** | Verify the accelerator |
| **3 — Configure** | GQA · MLA · MoE · Sliding-Window configs — all knobs documented |
| **4 — LR Finder** | Find the optimal learning rate before training *(optional)* |
| **5 — Train** | Streams dataset from HuggingFace Hub; supports resume + W&B |
| **6 — Generate** | Temperature · top-k · top-p · greedy · streaming · batch |
| **7 — Hub** | Push/pull checkpoint *(optional)* |
| **8 — LoRA** | Fine-tune with adapters *(optional)* |
| **9 — Benchmark** | Prefill latency + decode throughput |
| **10 — Plots** | 16 figures: perf · insights · 3D surfaces |

---
## 1 — Install

We upgrade JAX to the CUDA 12 build (Colab ships an older CPU-only version) and install DantinoX from PyPI.

> ⚠️ **Restart the runtime once after this cell** (`Runtime → Restart session`), then continue from cell 2.

In [ ]:
# Upgrade JAX + Flax to CUDA 12 build — must restart runtime after this cell.
# Required: Colab's default JAX is too old for its current CUDA plugin
# (PJRT_FFI version mismatch → JaxRuntimeError on the first GPU call).
# !pip install -q -U "jax[cuda12]" jaxlib flax

# Install / upgrade DantinoX + all optional extras (data, hub, benchmark)
!pip install -q -U "dantinox[data,hub,benchmark]"

print("\n✅ Done — restart the runtime now (Runtime → Restart session), then run from the next cell.")

---
## 2 — Check GPU

In [ ]:
import flax
import jax

print(f"JAX  version: {jax.__version__}")
print(f"Flax version: {flax.__version__}")

In [ ]:
import jax

devices = jax.devices()
print("JAX devices:", devices)

if not any("cuda" in str(d).lower() or "gpu" in str(d).lower() for d in devices):
    print("\n⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"\n✅ Running on {devices[0]}")

---
## 3 — Configure

`Config` exposes every knob: architecture, positional encoding, attention variant, MoE, tokenizer, dataset, optimiser, scheduler, regularisation, LoRA, and multi-GPU sharding.

The cells below define four ready-to-use configurations:

| Config | Attention | Extra | GPU target |
|---|---|---|---|
| `config_gqa` | GQA (baseline) | — | T4 / A100 |
| `config_mla` | MLA (DeepSeek-style) | latent KV | T4 / A100 |
| `config_moe` | GQA | Mixture-of-Experts MLP | A100 |
| `config_swa` | Sliding-Window | — | T4 |

Pick one and assign it to `config` at the bottom of the section.

In [ ]:
from dantinox import Config

# ── GQA baseline — trains in ~5 min on T4 ────────────────────────────────────
config_gqa = Config(
    # ── Architecture ─────────────────────────────────────────────────────────
    dim=256,             # model width (embedding + hidden size)
    n_heads=8,           # number of query heads
    head_size=32,        # per-head dimension  (dim == n_heads * head_size)
    num_blocks=6,        # transformer depth
    max_context=256,     # maximum sequence length
    kv_heads=2,          # GQA: 4 query heads share each KV head (set == n_heads for MHA)
    weight_tying=True,   # tie embedding and output projection weights

    # ── MLP ──────────────────────────────────────────────────────────────────
    use_swiglu=True,     # SwiGLU activation (better than GELU for language)
    expansion=4,         # MLP hidden = dim * expansion
    activation="gelu",   # used only when use_swiglu=False: "gelu" | "relu" | "silu"

    # ── Normalisation ─────────────────────────────────────────────────────────
    norm_type="rmsnorm", # "rmsnorm" (faster) | "layernorm"

    # ── Positional encoding ───────────────────────────────────────────────────
    use_rotary_pos=True, # RoPE (recommended)
    rope_scale_factor=1.0, # >1 compresses RoPE frequencies for longer contexts (NTK-aware)
    trainable_pos=False, # learnable absolute bias (combines with RoPE when True)
    absolute_pos=False,  # simple absolute positional embeddings (legacy)

    # ── Attention ────────────────────────────────────────────────────────────
    use_flash_attention=True,   # memory-efficient attention kernel
    sliding_window=False,       # restrict attention to a local window (see config_swa)

    # ── Regularisation ───────────────────────────────────────────────────────
    dropout_rate=0.1,
    gradient_checkpointing=True, # recompute activations on backward (saves VRAM)

    # ── Dataset — streamed from HuggingFace ──────────────────────────────────
    dataset_source="huggingface",  # "huggingface" | "local"
    dataset_name="Daniele/dante-corpus",
    dataset_config="",             # HF subset, e.g. "en" for allenai/c4
    dataset_text_field="text",
    dataset_split="train",
    streaming=True,                # stream from Hub without downloading

    # ── Tokenizer ────────────────────────────────────────────────────────────
    tokenizer_type="char",   # "char" (fast) | "bpe" (better compression)
    # tokenizer_path=None,   # path to a pre-trained BPE tokenizer JSON

    # ── Optimiser ────────────────────────────────────────────────────────────
    optimizer="adamw",   # "adamw" | "adam" | "lion" | "adafactor"
    lr=3e-4,
    lr_schedule="wsd",   # "wsd" (warmup→stable→cosine) | "cosine" | "linear" | "constant"
    warmup_steps=100,
    grad_clip=1.0,
    use_bf16=True,       # bfloat16 mixed precision (faster + less VRAM on Ampere/T4)

    # ── Training loop ─────────────────────────────────────────────────────────
    epochs=300,
    batch_size=64,
    grad_accum=4,        # effective batch = batch_size * grad_accum = 256
    patience=10,         # early stopping: stop if val loss doesn't improve for N epochs
    seed=42,

    # ── Evaluation ───────────────────────────────────────────────────────────
    eval_iters=20,       # batches averaged for each val-loss estimate

    # ── Multi-GPU (optional) ─────────────────────────────────────────────────
    n_devices=0,         # 0 = use all available GPUs; set e.g. 2 to limit
)

print("GQA config:"); print(config_gqa)

In [ ]:
# ── MLA — Multi-head Latent Attention (DeepSeek-style KV compression) ────────
# KV cache is compressed to a low-rank latent space → much smaller memory
# footprint than GQA at the same parameter count.
config_mla = Config(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=256,

    # MLA settings
    mla=True,            # enable Multi-head Latent Attention
    down_dim_kv=64,      # latent KV dim (≪ dim saves cache; try 32–128)
    down_dim_q=128,      # latent Q  dim (set to dim for full quality)
    rope_dim=32,         # RoPE is applied only to this many dims per head
                         # (remaining dims go through the low-rank path)

    kv_heads=1,          # with MLA usually 1 — the latent absorbs the sharing
    use_swiglu=True, norm_type="rmsnorm", use_flash_attention=True,
    weight_tying=True, dropout_rate=0.1,

    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True,

    optimizer="adamw", lr=3e-4, lr_schedule="wsd", warmup_steps=100,
    grad_clip=1.0, use_bf16=True,
    epochs=300, batch_size=64, grad_accum=4, patience=10, seed=42,
)

print("MLA config:"); print(config_mla)

In [ ]:
# ── MoE — Mixture of Experts MLP (sparse activation) ─────────────────────────
# Each token activates only top_k_mlp of n_experts MLP sub-networks.
# Parameters scale with n_experts but FLOPS stay roughly constant.
# Needs more VRAM — recommended for A100.
config_moe = Config(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=256, kv_heads=2,

    # MoE settings
    use_moe=True,        # replace dense MLP with a mixture-of-experts MLP
    n_experts=4,         # total number of expert sub-networks
    top_k_mlp=2,         # experts activated per token (must be ≤ n_experts)
    alpha_balance=0.1,   # load-balancing loss weight (prevents expert collapse)
    expansion=4,         # each expert's hidden = dim * expansion

    use_swiglu=True, norm_type="rmsnorm", use_flash_attention=True,
    weight_tying=True, dropout_rate=0.1,

    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True,

    optimizer="adamw", lr=3e-4, lr_schedule="wsd", warmup_steps=100,
    grad_clip=1.0, use_bf16=True,
    epochs=300, batch_size=32, grad_accum=8, patience=10, seed=42,
)

print("MoE config:"); print(config_moe)

In [ ]:
# ── Sliding-Window Attention — O(n · w) instead of O(n²) ─────────────────────
# Each token attends only to the last `context_window` tokens.
# Enables very long sequences on a T4 without OOM.
config_swa = Config(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=512, kv_heads=2,

    # Sliding-window settings
    sliding_window=True, # enable local attention
    context_window=64,   # each token sees the previous 64 tokens
    no_sink=True,        # True = exclude sink-token trick; False = keep first token visible

    use_swiglu=True, norm_type="rmsnorm", use_flash_attention=True,
    weight_tying=True, dropout_rate=0.1,

    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True,

    optimizer="lion",    # Lion often converges faster than AdamW
    lr=1e-4,             # Lion needs a lower LR than AdamW (~3–10× lower)
    lr_schedule="cosine", warmup_steps=200,
    grad_clip=1.0, use_bf16=True,
    epochs=300, batch_size=64, grad_accum=4, patience=10, seed=42,
)

print("SWA config:"); print(config_swa)

In [ ]:
# ── Pick the config to train ──────────────────────────────────────────────────
config = config_gqa   # swap to config_mla / config_moe / config_swa

---
## 4 — LR Finder *(optional)*

Sweeps the learning rate exponentially over `num_steps` mini-batches and plots loss vs LR.
Run this before a long training run to confirm the LR in your config is well-placed.

In [ ]:
import matplotlib.pyplot as plt

from dantinox import Trainer

suggested_lr, lrs, losses = Trainer(config).find_lr(
    num_steps=100,  # more steps → smoother curve; 100 is enough for a quick check
    min_lr=1e-7,
    max_lr=1.0,
    smoothing=0.9,
)
print(f"Suggested LR: {suggested_lr:.2e}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lrs, losses)
ax.axvline(suggested_lr, color="red", linestyle="--", label=f"suggested {suggested_lr:.2e}")
ax.set_xscale("log"); ax.set_xlabel("LR"); ax.set_ylabel("Smoothed loss")
ax.set_title("LR Range Test"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Optionally update the config with the found LR
# config.lr = suggested_lr

---
## 5 — Train

`Trainer.fit()` streams data, trains, evaluates, applies early stopping, and saves the checkpoint automatically.

In [ ]:
from dantinox import Trainer

run_dir = Trainer(config).fit(
    # data_path=None,        # omit to stream from HuggingFace (set in config)
    # run_dir="runs/my_run", # fix the output directory; auto-generated if omitted
    # resume=True,           # continue from the last saved checkpoint in run_dir
    # wandb_project="dantinox-experiments",  # log metrics to Weights & Biases
)

print(f"\nCheckpoint saved to: {run_dir}")

---
## 6 — Generate

`Generator` supports single prompts, batched prompts, and streaming — each with independent sampling controls.

In [ ]:
from dantinox import Generator

gen = Generator(
    run_dir,
    seed=42,      # controls sampling randomness; change for different outputs
)

prompt = "Nel mezzo del cammin "

# ── Sampling strategies ───────────────────────────────────────────────────────

# 1. Temperature sampling — higher = more creative, lower = more focused
print("=== temperature=0.8, top_k=40 ===")
print(gen.generate(prompt, max_new_tokens=120, temperature=0.8, top_k=40))

# 2. Nucleus (top-p) sampling — keeps the smallest set of tokens whose
#    cumulative probability ≥ top_p; adapts vocabulary size dynamically
print("\n=== top_p=0.9 ===")
print(gen.generate(prompt, max_new_tokens=120, temperature=1.0, top_p=0.9))

# 3. Greedy decoding — always picks the most likely token (deterministic)
print("\n=== greedy ===")
print(gen.generate(prompt, max_new_tokens=120, greedy=True))

# 4. Combined top-k + top-p + temperature (common production setting)
print("\n=== top_k=50, top_p=0.95, temperature=0.7 ===")
print(gen.generate(prompt, max_new_tokens=200, top_k=50, top_p=0.95, temperature=0.7))

In [ ]:
# ── Batch generation — one forward pass ──────────────────────────
prompts = [
    "Nel mezzo del cammin ",
    "Lasciate ogni speranza ",
    "Per me si va nella città dolente ",
]

outputs = gen.generate_batch(prompts, max_new_tokens=80, temperature=0.9)
for p, o in zip(prompts, outputs):
    print(f"▶ {p!r}")
    print(o)
    print()

In [ ]:
# ── Streaming — tokens printed as they arrive ─────────────────────
print("Streaming: ", end="", flush=True)
for chunk in gen.stream("Nel mezzo del cammin ", max_new_tokens=150, temperature=0.8):
    print(chunk, end="", flush=True)
print()

---
## 6 — HuggingFace Hub *(optional)*

Push your checkpoint so anyone can load it with `Generator("your-name/your-model")`.

You need a [HuggingFace account](https://huggingface.co/join) and a write token.

In [ ]:
HF_TOKEN  = "hf_..."          # paste your HF token here
HF_REPO   = "your-name/dantinox-dante"   # repo to create / update

from dantinox import push

url = push(run_dir, HF_REPO, token=HF_TOKEN, private=False)
print(f"Pushed to: {url}")

In [ ]:
# Load directly from the Hub on any machine — no pull step needed
gen_hub = Generator(HF_REPO, token=HF_TOKEN)
print(gen_hub.generate("Nel mezzo del cammin ", max_new_tokens=100))

---
## 8 — LoRA Fine-Tuning *(optional)*

Fine-tune with Low-Rank Adapters — only ~0.2 % of parameters are trained while the base weights stay frozen.

| Parameter | What it controls |
|---|---|
| `lora_rank` | adapter bottleneck width — higher = more capacity but more memory |
| `lora_alpha` | scales the adapter output: effective LR = lr × alpha / rank |
| `lora_dropout` | dropout applied inside the adapter (regularisation) |
| `lora_targets` | which modules to adapt: `"attention"` \| `"mlp"` \| `"all"` |

In [ ]:
import os

from dantinox import Config, Generator, Trainer

# Load the base config and switch on LoRA
ft_config = Config.from_yaml(os.path.join(run_dir, "config.yaml"))

ft_config.use_lora     = True
ft_config.lora_rank    = 8      # rank 4–16 covers most use cases
ft_config.lora_alpha   = 16.0   # keep alpha == 2 * rank as a starting point
ft_config.lora_dropout = 0.05   # small dropout helps with short fine-tuning corpora
ft_config.lora_targets = "attention"  # "attention" | "mlp" | "all"

# Fine-tuning schedule — shorter run, lower LR than pre-training
ft_config.epochs        = 50
ft_config.lr            = 1e-4   # 3–10× lower than pre-training LR
ft_config.lr_schedule   = "cosine"
ft_config.warmup_steps  = 20
ft_config.patience      = 5

# Swap dataset if adapting to a different domain
ft_config.dataset_name = "Daniele/dante-corpus"   # replace with your corpus

ft_run = Trainer(ft_config).fit()
print(f"\nLoRA checkpoint: {ft_run}")

# Generate from the fine-tuned model
ft_gen = Generator(ft_run, seed=42)
print(ft_gen.generate("Nel mezzo del cammin ", max_new_tokens=200, temperature=0.8))

---
## 8 — Benchmark

`BenchmarkRunner` measures:
- **Prefill latency** — time (ms) to process the prompt in a single forward pass
- **Decode throughput** — tokens/second at sequence lengths `[64, 128, 256]` (BS=1)
- **Batch throughput** — tokens/second at batch sizes `[1, 4, 16, 64]` (seq=256)
- **FLOPs** — XLA cost analysis for both prefill and decode steps
- **Theoretical KV-cache size** — bytes per layer at `max_context`

Results are saved to `benchmark_results.csv` and returned as a DataFrame.

In [ ]:
import os

from dantinox import BenchmarkRunner

# Keep seq-lens and batch sizes small for Colab's T4 (16 GB VRAM)
runner = BenchmarkRunner(
    "runs",
    seq_lens=[64, 128, 256],
    batch_sizes=[1, 4, 16, 64],
)

# Benchmark only the run we just trained (pass the folder name, not the full path)
run_name = os.path.basename(run_dir)
df = runner.run(run_names=[run_name], out_csv="benchmark_results.csv")

# Pretty-print the key columns
cols = [
    "run", "type", "params_m", "theoretical_cache_mb",
    "prefill_ms", "tps_64", "tps_128", "tps_256",
    "tps_bs1", "tps_bs4", "tps_bs16", "tps_bs64",
    "decode_gflops", "val_loss",
]
display(df[[c for c in cols if c in df.columns]].T)

In [ ]:
# ── Benchmark ALL runs in the runs/ directory ─────────────────────
# Useful if you trained multiple configs (e.g., after LoRA fine-tuning)
df_all = runner.run(out_csv="benchmark_results_all.csv")
print(f"Benchmarked {len(df_all)} run(s)")
display(df_all[["run", "type", "params_m", "prefill_ms", "tps_256", "val_loss"]])

---
## 9 — Plots

`Plotter` generates charts from the benchmark CSV — no extra setup needed.

| Group | What it shows |
|---|---|
| `perf` | Decode throughput vs seq-len · Batch throughput vs batch-size · Prefill latency |
| `3d` | 3-D surface: throughput × seq-len × batch-size |
| `insights` | Quality vs throughput · Cache size vs throughput (meaningful with ≥2 runs) |

In [ ]:
from dantinox import Plotter

saved = Plotter("benchmark_results.csv", out_dir="plots").run()
for group, path in saved.items():
    print(f"{group}: {path}")

In [ ]:
# Display all saved PNGs inline
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

for group, path in saved.items():
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(14, 5) if group == "perf" else (8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(group)
    plt.tight_layout()
    plt.show()

---
## Tips

### A100 large model (GQA)

```python
config = Config(
    dim=512, n_heads=16, head_size=32, num_blocks=12,
    max_context=512, kv_heads=4,
    use_swiglu=True, norm_type="rmsnorm",
    use_flash_attention=True, weight_tying=True,
    gradient_checkpointing=True, dropout_rate=0.1,
    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True,
    optimizer="adamw", lr=3e-4, lr_schedule="wsd",
    warmup_steps=500, grad_clip=1.0, use_bf16=True,
    epochs=1000, batch_size=128, grad_accum=8, patience=20, seed=42,
)
```

### A100 large MLA model

```python
config = Config(
    dim=512, n_heads=16, head_size=32, num_blocks=12,
    max_context=512,
    mla=True, down_dim_kv=128, down_dim_q=256, rope_dim=32, kv_heads=1,
    use_swiglu=True, norm_type="rmsnorm", use_flash_attention=True,
    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True,
    optimizer="adamw", lr=3e-4, lr_schedule="wsd",
    warmup_steps=500, grad_clip=1.0, use_bf16=True,
    epochs=1000, batch_size=128, grad_accum=8, patience=20, seed=42,
)
```

### Different datasets

| Dataset | `dataset_name` | `dataset_config` | `dataset_text_field` |
|---|---|---|---|
| WikiText-103 | `wikitext` | `wikitext-103-v1` | `text` |
| TinyStories | `roneneldan/TinyStories` | *(empty)* | `text` |
| OpenWebText | `openwebtext` | *(empty)* | `text` |
| C4 (English) | `allenai/c4` | `en` | `text` |
| Local file | set `dataset_source="local"` | — | `dataset_name="path/to/corpus.txt"` |

### BPE tokenizer (better compression than char)

```python
config.tokenizer_type = "bpe"
config.vocab_size     = 8000   # ~8k is good for Italian; 32k for English
```

### Resume an interrupted run

```python
run_dir = Trainer(config).fit(resume=True, run_dir="runs/run_20260101_120000")
```

### Log to Weights & Biases

```python
run_dir = Trainer(config).fit(wandb_project="dantinox-experiments")
```

### LR finder before a long run

```python
suggested_lr, lrs, losses = Trainer(config).find_lr(num_steps=100)
config.lr = suggested_lr
```

### Benchmark and compare all trained configs

```python
from dantinox import BenchmarkRunner, Plotter

df = BenchmarkRunner("runs").run(out_csv="benchmark_results.csv")
Plotter("benchmark_results.csv", out_dir="plots").run()
```

### Run only specific plot groups

```python
Plotter("benchmark_results.csv").run(groups=["perf", "insights"])
# groups: "perf" | "insights" | "3d" | "3d_dkv"
```